experiment with pruning DOM for better results

some code cribbed from https://github.com/bytetunnels/bytetunnels.github.com/blob/master/_posts/2026-02-20-llm-powered-data-extraction-schema-driven-scraping-with-structured-output.md

In [10]:
from bs4 import BeautifulSoup, Comment
import re
import requests

In [32]:


def prune_dom(html: str, content_selector: str = None) -> str:
    """
    Prune HTML to reduce token count while preserving extractable content.
    Inspired by the AXE paper's approach to intelligent DOM reduction.
    """
    soup = BeautifulSoup(html, "html.parser")

    # Stage 1: Remove elements that never contain useful data
    tags_to_remove = [
        "script", "style", "noscript", "iframe", "svg",
        "link", "meta", "header", "footer", "nav",
    ]
    for tag in tags_to_remove:
        for element in soup.find_all(tag):
            element.decompose()

    # Stage 2: Remove hidden elements
    for element in soup.find_all(attrs={"style": re.compile(r"display:\s*none")}):
        element.decompose()
    for element in soup.find_all(attrs={"hidden": True}):
        element.decompose()
    for element in soup.find_all(attrs={"aria-hidden": "true"}):
        element.decompose()

    # Stage 3: Remove common noise by class/id patterns
    noise_patterns = re.compile(
        r"(cookie|consent|popup|modal|overlay|sidebar|widget|social|share|newsletter|subscribe|advert|tracking)",
        re.IGNORECASE,
    )
    for element in soup.find_all(
        attrs={"class": noise_patterns}
    ):
        element.decompose()
    for element in soup.find_all(
        attrs={"id": noise_patterns}
    ):
        element.decompose()

    # Stage 4: Focus on main content if a selector is provided
    if content_selector:
        main_content = soup.select_one(content_selector)
        if main_content:
            soup = BeautifulSoup(str(main_content), "html.parser")

    # Stage 5: Strip unnecessary attributes
    for element in soup.find_all(True):
        allowed_attrs = {"href", "src", "alt", "title", "class", "id"}
        attrs_to_remove = [
            attr for attr in element.attrs if attr not in allowed_attrs
        ]
        for attr in attrs_to_remove:
            del element[attr]

    # Stage 5.5 (added by me): remove comments
    comments = soup.find_all(string=lambda text: isinstance(text, Comment))
    for comment in comments:
        comment.extract()

    # Stage 6: Collapse whitespace
    text = str(soup)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r">\s+<", "><", text)

    return text.strip()


# # Usage
# import requests
# raw_html = requests.get("https://example.com/products").text
# print(f"Original: {len(raw_html)} characters")

# pruned = prune_dom(raw_html, content_selector="main")
# print(f"Pruned: {len(pruned)} characters")
# print(f"Reduction: {(1 - len(pruned)/len(raw_html))*100:.1f}%")

In [3]:
raw_html = requests.get("https://www.sportsbookreview.com/betting-odds/mlb-baseball/?date=2026-06-03").text

In [6]:
len(raw_html)

606293

In [4]:
pruned = prune_dom(raw_html, content_selector="#section-mlb")

In [5]:
len(pruned)

171081

In [33]:
pruned3 = prune_dom(raw_html, content_selector="#section-mlb")

In [34]:
len(pruned3)

170383

In [40]:
soup3 = BeautifulSoup(pruned3, "html.parser")

In [ ]:
print(soup3.prettify())

this should work with DocumentScraperGraph

In [38]:
from scrapegraphai.graphs import DocumentScraperGraph
from scrapers import sbr_mlb_scraper

In [41]:
graph_config = {
    "llm": {
        #"model": "ollama/llama3.2", # this causes my computer to melt down, and hallucinates
        "model": "ollama/llama3.2:1b", # runs ok, but still hallucinates
        
        "model_tokens": 8192,
        "format": "json",
    },
    "verbose": True,
    "headless": True,
}

graf = DocumentScraperGraph(
    prompt="Extract structured betting data from this HTML.",
    config=graph_config,
    source=str(soup3),
    schema = sbr_mlb_scraper.MLBGames
)

this takes FOREVER to run even with the 1B param version of llama3.2

In [ ]:
result = graf.run()

--- Executing Fetch Node ---
--- Executing ParseNode Node ---
--- Executing GenerateAnswer Node ---
Processing chunks: 100%|██████████| 14/14 [00:00<00:00, 32334.94it/s]
